## Chapter 2 Hydraulics QC/Review Tool

This tool scans through slope maps and calculations allowing the reviewer to flag suspect river reaches

In [1]:
"""
River Hydraulic Geometry QC Review Tool

This script displays reach maps and slope regression figures side-by-side
for visual quality control review. User classifies each river as "Good" 
or "Needs work", and results are saved to a CSV file.

Usage:
1. Set the working_directory and output_csv_path variables
2. Run all cells
3. Review each river by clicking "Good" or "Needs work" buttons
4. Results are automatically saved to CSV when complete
"""

import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Image, clear_output
from PIL import Image as PILImage
import io

# ===== CONFIGURATION =====
# Set your working directory where HydraulicGeometry folder is located
working_directory = r"E:\Dissertation\Data"

# Set output path for the QC results CSV
output_csv_path = os.path.join(working_directory, "RiverMapping", "HydraulicGeometry", "river_QC_review.csv")

# ===========================


class RiverQCReviewer:
    def __init__(self, working_directory, output_csv_path):
        self.working_directory = working_directory
        self.output_csv_path = output_csv_path
        self.hydraulic_geom_dir = os.path.join(working_directory, "RiverMapping", "HydraulicGeometry")
        
        # Find all rivers
        self.rivers = self.find_rivers()
        self.current_index = 0
        self.results = []
        
        # Create widgets
        self.output = widgets.Output()
        self.good_button = widgets.Button(
            description='Good',
            button_style='success',
            tooltip='Mark this river as good quality',
            icon='check',
            layout=widgets.Layout(width='150px', height='50px')
        )
        self.needs_work_button = widgets.Button(
            description='Needs Work',
            button_style='warning',
            tooltip='Mark this river as needing further review',
            icon='exclamation-triangle',
            layout=widgets.Layout(width='150px', height='50px')
        )
        self.status_label = widgets.HTML(value="")
        
        # Button callbacks
        self.good_button.on_click(lambda b: self.record_result("Good"))
        self.needs_work_button.on_click(lambda b: self.record_result("Needs work"))
        
    def find_rivers(self):
        """Find all river directories with slope regression outputs"""
        rivers = []
        
        if not os.path.exists(self.hydraulic_geom_dir):
            print(f"Error: Directory not found: {self.hydraulic_geom_dir}")
            return rivers
        
        for river_name in os.listdir(self.hydraulic_geom_dir):
            river_path = os.path.join(self.hydraulic_geom_dir, river_name)
            
            # Skip files, only process directories
            if not os.path.isdir(river_path):
                continue
            
            # Check if Slope_regressions folder exists
            slope_dir = os.path.join(river_path, "Slope_regressions")
            if os.path.exists(slope_dir):
                # Find map and profile images
                images = os.listdir(slope_dir)
                maps = [img for img in images if '_map.png' in img]
                profiles = [img for img in images if '_profile.png' in img]
                
                if maps and profiles:
                    rivers.append({
                        'name': river_name,
                        'slope_dir': slope_dir,
                        'maps': sorted(maps),
                        'profiles': sorted(profiles)
                    })
        
        return sorted(rivers, key=lambda x: x['name'])
    
    def combine_images(self, map_path, profile_path):
        """Combine two images side by side"""
        try:
            img_map = PILImage.open(map_path)
            img_profile = PILImage.open(profile_path)
            
            # Calculate dimensions for side-by-side display
            total_width = img_map.width + img_profile.width
            max_height = max(img_map.height, img_profile.height)
            
            # Create new image
            combined = PILImage.new('RGB', (total_width, max_height), 'white')
            combined.paste(img_map, (0, 0))
            combined.paste(img_profile, (img_map.width, 0))
            
            # Resize if too large (max width 1400px)
            if total_width > 1400:
                ratio = 1400 / total_width
                new_width = 1400
                new_height = int(max_height * ratio)
                combined = combined.resize((new_width, new_height), PILImage.Resampling.LANCZOS)
            
            # Convert to bytes for display
            img_bytes = io.BytesIO()
            combined.save(img_bytes, format='PNG')
            img_bytes.seek(0)
            
            return img_bytes.read()
        except Exception as e:
            print(f"Error combining images: {e}")
            return None
    
    def display_current_river(self):
        """Display the current river's images"""
        with self.output:
            clear_output(wait=True)
            
            if self.current_index >= len(self.rivers):
                self.finish_review()
                return
            
            river = self.rivers[self.current_index]
            
            # Update status
            progress = f"River {self.current_index + 1} of {len(self.rivers)}"
            self.status_label.value = f"<h3>{river['name']}</h3><p>{progress}</p>"
            
            # Display all reaches for this river
            print(f"\n{'='*80}")
            print(f"River: {river['name']} ({progress})")
            print(f"{'='*80}\n")
            
            for i, (map_img, profile_img) in enumerate(zip(river['maps'], river['profiles'])):
                reach_num = i + 1
                print(f"\nReach {reach_num}:")
                print("-" * 40)
                
                map_path = os.path.join(river['slope_dir'], map_img)
                profile_path = os.path.join(river['slope_dir'], profile_img)
                
                combined_img = self.combine_images(map_path, profile_path)
                
                if combined_img:
                    display(Image(data=combined_img))
                else:
                    print(f"Could not load images for reach {reach_num}")
            
            print(f"\n{'='*80}")
            print("Review all reaches above, then click 'Good' or 'Needs Work'")
            print(f"{'='*80}\n")
    
    def record_result(self, status):
        """Record the result and move to next river"""
        if self.current_index < len(self.rivers):
            river = self.rivers[self.current_index]
            self.results.append({
                'river_name': river['name'],
                'status': status,
                'num_reaches': len(river['maps'])
            })
            
            # Save to CSV after each classification (in case of interruption)
            self.save_results()
            
            # Move to next river
            self.current_index += 1
            self.display_current_river()
    
    def save_results(self):
        """Save results to CSV"""
        if self.results:
            df = pd.DataFrame(self.results)
            
            # Create output directory if needed
            os.makedirs(os.path.dirname(self.output_csv_path), exist_ok=True)
            
            df.to_csv(self.output_csv_path, index=False)
    
    def finish_review(self):
        """Called when all rivers have been reviewed"""
        with self.output:
            clear_output(wait=True)
            print("\n" + "="*80)
            print("🎉 REVIEW COMPLETE!")
            print("="*80)
            print(f"\nReviewed {len(self.results)} rivers")
            
            if self.results:
                df = pd.DataFrame(self.results)
                print(f"\nResults saved to: {self.output_csv_path}")
                print("\nSummary:")
                print(df['status'].value_counts())
                print("\n" + "="*80)
            
        self.status_label.value = "<h3>Review Complete! ✓</h3>"
        self.good_button.disabled = True
        self.needs_work_button.disabled = True
    
    def start(self):
        """Start the review process"""
        if not self.rivers:
            print("No rivers found with slope regression outputs!")
            print(f"Checked directory: {self.hydraulic_geom_dir}")
            return
        
        print(f"Found {len(self.rivers)} rivers to review\n")
        
        # Create layout
        button_box = widgets.HBox([
            self.good_button,
            widgets.Label("     "),  # Spacer
            self.needs_work_button
        ])
        
        ui = widgets.VBox([
            self.status_label,
            self.output,
            button_box
        ])
        
        # Display UI
        display(ui)
        
        # Show first river
        self.display_current_river()


# ===== START THE REVIEW =====
print("Initializing River QC Review Tool...")
print(f"Working directory: {working_directory}")
print(f"Output CSV: {output_csv_path}\n")

reviewer = RiverQCReviewer(working_directory, output_csv_path)
reviewer.start()

Initializing River QC Review Tool...
Working directory: E:\Dissertation\Data
Output CSV: E:\Dissertation\Data\RiverMapping\HydraulicGeometry\river_QC_review.csv

Found 132 rivers to review

